In [42]:
import os

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel
from langchain_groq import ChatGroq
load_dotenv()

True

Step 1a - Indexing (Document Ingestion)

In [ ]:
loader = PyMuPDFLoader(file_path="Apache Kafka Beginner to Advanced Guide.pdf")
documents = loader.load()

print("Total pages:",{len(documents)})

    

Total pages: {38}


Step 1b - Indexing(Text Splitting)

In [13]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap = 200)
chunks = splitter.split_documents(documents)

print("Total Chunks:", len(chunks))


Total Chunks: 86


Step 1c and 1d - Indexing(Embedding Generation and storing in vector store)

In [14]:
# 1. Set up paths and model
persist_directory = "./chroma_db"
embedding_model = os.getenv("OLLAMA_EMBEDDING_MODEL")
embeddings = OllamaEmbeddings(model = embedding_model)

if os.path.exists(persist_directory) and os.listdir(persist_directory):
    print("Loading existing Chroma database from disk...")
    vector_store = Chroma(
        persist_directory=persist_directory,
        embedding_function=embeddings,
        collection_name="kafka_pdf"
    )

else:
    print("Database not found. Creating embeddings and building database.")
    vectore_store = Chroma.from_documents(
        documents=chunks,
        embeddings=embeddings,
        collection_name="kafka_pdf",
        persist_directory=persist_directory
    )    
    print("Database Created and saved!")

print(f"Collection count: {vector_store._collection.count()}")    



Loading existing Chroma database from disk...
Collection count: 86


Step 2 - Retrieval

In [15]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k" : 3
    }
)

retriever

VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000022785AB6C60>, search_kwargs={'k': 3})

Step2 a. Retreiver Invoke


In [ ]:
# retriever.invoke("What is kafka")

[Document(id='f0faf496-56b1-454e-97f1-f1a14f56dce3', metadata={'trapped': '', 'modDate': "D:20261006064649+00'00'", 'moddate': '2026-10-06T06:46:49+00:00', 'creationDate': "D:20261006064649+00'00'", 'producer': 'Skia/PDF m153', 'file_path': 'Apache Kafka Beginner to Advanced Guide.pdf', 'author': '', 'format': 'PDF 1.7', 'creator': 'Chromium', 'keywords': '', 'source': 'Apache Kafka Beginner to Advanced Guide.pdf', 'title': 'Apache Kafka: Beginner to Advanced Guide', 'creationdate': '2026-10-06T06:46:49+00:00', 'page': 0, 'subject': '', 'total_pages': 38}, page_content='The problem it solves\nWithout a central log, every system talks directly to every other system. With N producers\nand M consumers you get N × M point-to-point integrations, each with its own format,\nretry logic and failure mode. Kafka replaces that mesh with one hub: producers write\nonce, and any number of consumers read independently, at their own pace.\nWhat an event is\nAn event (also called a record or message) i

Step3. Augmentation


In [18]:
template = """You are an expert technical assistant. Answer the question using ONLY the provided context below. 
If you do not know the answer based on the context, say "I don't know based on the provided documents."

Context:
{context}

Question: {question}"""

prompt = PromptTemplate(
    template = template,
    input_variables=['context','question']
)


In [19]:
question = "how can we use kafka in our application"
retriever_docs = retriever.invoke(question)
retriever_docs

[Document(id='834787d6-40d2-4125-b29f-6f50672baf96', metadata={'file_path': 'Apache Kafka Beginner to Advanced Guide.pdf', 'page': 0, 'format': 'PDF 1.7', 'source': 'Apache Kafka Beginner to Advanced Guide.pdf', 'trapped': '', 'author': '', 'producer': 'Skia/PDF m153', 'keywords': '', 'subject': '', 'modDate': "D:20261006064649+00'00'", 'moddate': '2026-10-06T06:46:49+00:00', 'total_pages': 38, 'creationdate': '2026-10-06T06:46:49+00:00', 'creationDate': "D:20261006064649+00'00'", 'creator': 'Chromium', 'title': 'Apache Kafka: Beginner to Advanced Guide'}, page_content='Apache Kafka: Beginner to Advanced Guide\n\u200bOct 6, 2026 · \u200b@E2E Data Solutions\nApache Kafka is a distributed, durable, append-only log that lets systems publish, store\nand process streams of events in real time. This guide takes you from first principles to\nproduction operations, in 16 sections. Beginners should read sections 1 to 6 in order.\nIntermediate readers can jump to sections 7 to 10. Advanced reade

In [23]:
content_text = "\n\n".join([doc.page_content for doc in retriever_docs])
content_text

'Apache Kafka: Beginner to Advanced Guide\n\u200bOct 6, 2026 · \u200b@E2E Data Solutions\nApache Kafka is a distributed, durable, append-only log that lets systems publish, store\nand process streams of events in real time. This guide takes you from first principles to\nproduction operations, in 16 sections. Beginners should read sections 1 to 6 in order.\nIntermediate readers can jump to sections 7 to 10. Advanced readers will find internals,\nsecurity and operations in sections 11 to 15.\n1. Introduction: what Kafka is and why it exists\nKafka was built at LinkedIn in 2010 to move activity data between hundreds of systems. It\nwas open-sourced in 2011 and became a top-level Apache project in 2012. Today it is the\ndefault backbone for real-time data in most large companies.\nThe problem it solves\nWithout a central log, every system talks directly to every other system. With N producers\nand M consumers you get N × M point-to-point integrations, each with its own format,\n\nKeep clie

In [29]:
final_prompt = prompt.invoke({
    "context" : content_text,
    "question" : question
})

Step4. Generation

In [43]:
groq_model = os.getenv("GROQ_MODEL")
llm = ChatGroq(model=groq_model)
result = llm.invoke(final_prompt)
print(result.content)

Based on the provided documents, you can use Apache Kafka in your application for the following common use cases:

*   **Real-time data pipelines:** Moving data from applications and databases into warehouses and lakes (such as Snowflake, BigQuery, S3, or Databricks).
*   **Change data capture (CDC):** Streaming every insert, update, and delete from a database (often using tools like Debezium).
*   **Event-driven microservices:** Allowing services to react to events rather than calling each other synchronously.
*   **Log and metrics aggregation:** Collecting logs from thousands of servers into a single location.
*   **Stream processing:** Handling tasks such as fraud detection, real-time recommendations, or IoT sensor alerts.
*   **Event sourcing:** Storing every state change as an event and rebuilding state by replaying those events.

Additionally, Kafka allows systems to publish, store, and process streams of events in real time. However, the documents note that you should **not** us